# RESTNET -- Residual Neural Network

## ResNet (Residual Network) is a type of deep Convolutional Neural Network (CNN) architecture that uses skip connections (shortcut connections) to make very deep neural networks easier to train.

## BEFORE RESTNET THE PEOPLE ASSUMED THAT MORE LAYER == MORE LEARNING 

### THEY ASSUMED AS 50 LAYER IS GREATER THAN 20 LAYER 

### BUT EVENTUALLY THE RESTNET CHANGED IT 

## AFTER REST NET PEOPLE CAME TO KNOW THAT MORE LAYER IS NOT EQUAL TO MORE LEARNING 

### AND ALSO 50 LAYER IS NOT ALWAYS GREATER THAN 20 LAYER  

### THE MAIN ISSUE WASN'T ONLY OVERFITTING 

### IT WAS ASLO OPTIMISATION TECHNIQUES

### NOW CONSIDER YOU HAVE A NETWORK OF 50 LAYER BUT THE ERROR IS OPTIMISED IN THE LAYER 20 ITSELF 

### NOW WHAT SHOULD WE DO , WE SHOULD DIRECTLY GIVE THE OUTPUT 

### BUT THE REMANING LAYERS AGAIN TRIES TO OPTIMISE AND CREATE SOME ERROR 

### THIS IS THE PROBLEM WE GET IS LARGE LAYERED NETWORKS

## DEGARADATION 

### --THE CASE WHERE MORE NO OF LAYER IN A NEURAL NETWORK CAUSES MORE TRAINING ERROR 

## DEGARDATION GAP

## --the difference in training error between a shallower network and a deeper (plain) counterpart that should, in theory, perform at least as well.

### -- DEGRADATION GAP = ERROR(DEEP NETWORK) - ERROR(SHALLOW NETWORK)

## RESIDUAL 

### --USUALLY IT MEANS THE DIFFRENCE BETWEEN ACTUAL VALUE - PREDICTED VALUE


### --HERE RESIDUAL IS DENOTED AD F(X)=H(X)-X 

### --WHERE H(X)-->  final output of the current ResNet block

### --WHERE X --> OUTPUT OF THE PREV LAYER 

### --F(X) --> RESIDUAL LEARNED BY THE CURRENT BLOCK


## RESIDUAL BLOCK 

### --BASICALLY IT CONTAINS SET OF CONV LAYER , ACTIVATION FUNCTION AND ETCC LIKE A NORMAL BLOCK 

### -- BUT IT BRANCHES THE **X** (OUTPUT FROM THE PREV BLOCK) INTO TWO , ONE MAIN BRANCH AND THE OTHER IS SKIPPING BRANCH

### --HERE THE LAYERS DOESNT CALCULATE OR MODIFY THE ENTIRE **X** , THE LAYERS ONLY GIVE'S THE CORRECTION OF THE **X**

### ---TWO BRANCHES MAIN , SKIPPING BRANCH 

### --MAIN BRANCH GOES INTO THE LAYERS AND FINDS THE F(X)

### -- THE SKIPPING BRANCH ,SKIPS THE LAYERS AND PRESERVE THE **X**

### --SOO THE **X** IS PRESERVED AND THE MAIN BRANCH ONLY CALCULATES THE ERROR AND MODIFES AND BRING AS A F(X)


### --SOO THE FINAL OUPUT H(X) = F(X) + X

## ALGO 

### THE NEURAL NETWORK IS SPLITED INTO MULTIPLE RESIDUAL BLOCK 

### IN THE EACH BLOCK THE **X** (OUPUT OF THE PREV BLOCK) IS BRANCHED INTO TWO

### TWO BRANCHES ---> MAIN AND SKIPPING BRANCH 

### THE MAIN BRANCH GOES INTO ALL THE LAYERS PRESENT IN A RESIDUAL BLOCK AND CALCULATES THE CORRECTION OR THE RESIDUAL **F(X)**

### THE SKIPPING BRANCH PRESERVES THE **X**

### SOO HERE X (OUPUT OF PREV BLOCK,INPUT OF CURRENT BLOCK) IIS PRESERVED AND THE F(X) THE CORRECTION OR RESDIUAL IS CALCULATED AND ADDED 

### USUALLY IN NORMAL CNN IN EVERY LAYER THE INPUT IS MODIFED AND LEARNED AGAIN, WHILE HEERE WE ONLY MODIFY ACC TO THE RESIDUAL  

### THE OUTPUT OF THE CURRENT RESIDUAL BLOCK H(X) = F(X) + X

### WHERE F(X) CALCULATES ONLY THE CORRECTION AND DOESNT RELEARN 

## RESTNET IMPLEMENTATION

In [ ]:


import torch
import torch.nn as nn


# --------------------------------------------------------------------------
# BASIC BLOCK  (used in ResNet-18 / ResNet-34)
# Two 3x3 convs. Used when network depth is relatively small.
# --------------------------------------------------------------------------
class BasicBlock(nn.Module):
    expansion = 1  # output channels = out_channels * expansion

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()

        # ---- Main branch: computes F(X) ----
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                                stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # ---- Skip connection ----
        # downsample = projection shortcut, only needed when shape of X
        # doesn't match shape of F(X) (stride != 1 or channel count changes)
        self.downsample = downsample

    def forward(self, x):
        identity = x                      # preserve X untouched

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)               # this is F(X)

        if self.downsample is not None:
            identity = self.downsample(x)  # project X to match F(X) shape

        out = out + identity              # H(X) = F(X) + X
        out = self.relu(out)

        return out


# --------------------------------------------------------------------------
# BOTTLENECK BLOCK  (used in ResNet-50 / 101 / 152)
# 1x1 -> 3x3 -> 1x1 convs. 1x1 layers reduce then restore channel depth,
# making the 3x3 conv cheaper while keeping the block expressive.
# --------------------------------------------------------------------------
class Bottleneck(nn.Module):
    expansion = 4  # output channels = out_channels * expansion

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()

        # ---- Main branch: computes F(X) ----
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion,
                                kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

        self.relu = nn.ReLU(inplace=True)

        # ---- Skip connection ----
        self.downsample = downsample

    def forward(self, x):
        identity = x                      # preserve X untouched

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)               # this is F(X)

        if self.downsample is not None:
            identity = self.downsample(x)  # project X to match F(X) shape

        out = out + identity              # H(X) = F(X) + X
        out = self.relu(out)

        return out


# --------------------------------------------------------------------------
# FULL RESNET ARCHITECTURE
# --------------------------------------------------------------------------
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000, in_channels=3):
        """
        block      -> BasicBlock or Bottleneck
        layers     -> list of 4 ints, e.g. [2,2,2,2] for ResNet-18,
                      how many residual blocks go in each of the 4 stages
        num_classes-> number of output classes
        """
        super().__init__()
        self.in_channels = 64

        # ---- Stem: initial conv before any residual blocks ----
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # ---- 4 stages, each a stack of residual blocks ----
        # Each block's H(X) becomes the X for the next block
        self.layer1 = self._make_stage(block, 64, layers[0], stride=1)
        self.layer2 = self._make_stage(block, 128, layers[1], stride=2)
        self.layer3 = self._make_stage(block, 256, layers[2], stride=2)
        self.layer4 = self._make_stage(block, 512, layers[3], stride=2)

        # ---- Head ----
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        self._init_weights()

    def _make_stage(self, block, out_channels, num_blocks, stride):
        """Builds one stage = a stack of `num_blocks` residual blocks."""
        downsample = None

        # projection shortcut needed if shape changes at the start of the stage
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion)
            )

        layers_list = []
        # first block in the stage may change shape (stride/channels)
        layers_list.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion

        # remaining blocks: shape already matches, no downsample needed
        for _ in range(1, num_blocks):
            layers_list.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers_list)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)

        x = self.layer1(x)   # X -> H(X), which becomes X for layer2
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


# --------------------------------------------------------------------------
# STANDARD RESNET VARIANTS
# --------------------------------------------------------------------------
def resnet18(num_classes=1000):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

def resnet34(num_classes=1000):
    return ResNet(BasicBlock, [3, 4, 6, 3], num_classes)

def resnet50(num_classes=1000):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes)

def resnet101(num_classes=1000):
    return ResNet(Bottleneck, [3, 4, 23, 3], num_classes)

def resnet152(num_classes=1000):
    return ResNet(Bottleneck, [3, 8, 36, 3], num_classes)


# --------------------------------------------------------------------------
# QUICK TEST
# --------------------------------------------------------------------------
if __name__ == "__main__":
    x = torch.randn(2, 3, 224, 224)   # batch of 2 RGB images, 224x224

    for name, model_fn in [
        ("ResNet-18", resnet18),
        ("ResNet-34", resnet34),
        ("ResNet-50", resnet50),
    ]:
        model = model_fn(num_classes=10)
        out = model(x)
        num_params = sum(p.numel() for p in model.parameters())
        print(f"{name:10s} -> output shape: {tuple(out.shape)} | params: {num_params:,}")

# MOBILE NET ARCHITECTURE 

### BEFORE GOING INTO THIS ARCH WE WILL LEARN THE 2 DIFFRENT JOBS DONE IN EACH STEP OF THE FILETR 

### LET US ASSUME THE INPUT IMAGE HAS THREE INPUT CHANNELS (RED , GREEN AND BLUE)

### JOB 1 --> SPATIAL FILTERING

### --TO DO THE SUM OF PRODUCT OPERATION WITH IN THE CHANNEL AT EACH POSTION OF THE FILTER 
### LIKE WE DO SOP OPERATION WITH IN EACH CHANNEL LIKE RED , GREEN,BLUE AND THREE VALUES IN THREE CHANNELS EACH VALUE IN EACH CHANNEL

### JOB 2 --> CHANNEL MIXING 

### --- NEXT THEY DO HAVE THREE DIFFRENT VALUE OF SUM OF PRODUCT (AS WE DO HAVE THREE CHNANNELS REG,GREEN,BLUE) SOO THE NEXT JOB IS TO SUM IT UP ONCE AGAIN ACROSS THE THREE CHANELS AND BRING INTO A SINGLE VALUE 

### --SUM IT UP THE SOP VALUE'S ACROSS THE CHANNELS AND MAKE INTO A SINGLE VALUE 

### SOO AT EACH STEP OR FILTER WE DO TWO DIFFRENT JOB'S

## USUALLY IN NORMAL CNN , WE DO THE BOTH STEPS SIMENTUNALSELY 

### --THIS MAKE THE COMPUTATINAL COST MORE HIGHER 

## MOBILE NET MAKE'S THE PROCESS ONE BY ONE 

### INSTEAD OF CHANNEL MIXING + SPATIAL FILTERING IN A STEP , WE DO FIRST SPATIAL FILTERING FIRST THEN CHANNEL MIXING 

### FIRST WE WILL START FROM SPATIAL FILTERING 

### AS WE DONT DO CHANNEL MIXING AT THE VERY NEXT STEP 

### WE DO GET ONE FEATURE MAP FOR EACH CHANNEL 

### N CHANNEL ----> N FEATURE MAP'S